# Load dependencies

In [155]:
import os
from dotenv import load_dotenv
from supporting_functions import load_json, save_json, create_book_content_html_and_serve_with_flask, get_parsed_html_content, find_highlights_for_chapter
from html_functions import KindleHTMLParser, create_html
from tinydb import TinyDB, Query
from tinydb.table import Document
import shutil
main_db = TinyDB('./data/db/main.json')
kindle_highlights_db = TinyDB('./data/db/kindle_highlights.json')

# Load all the files

In [5]:
def create_book_id(book_name, db):
    base_id = book_name.lower().replace(" ", "-")
    book_id = base_id
    counter = 1

    # Check if the ID already exists in the database
    while db.contains(Query().id == book_id):
        book_id = f"{base_id}-{counter}"
        counter += 1

    return book_id

In [2]:
book_name = "The Invisible Empire"
temp_book_folder = "./books/invisible-empire/"
book_source = temp_book_folder + "book.epub"
highlights_source = temp_book_folder + "highlights.html"

In [24]:
def copy_and_rename_file(source_path, destination_dir, new_name):
    """
    Copy a file to a destination directory and rename it.
    If a file with the new name already exists, ask the user if they want to delete it.

    Args:
        source_path (str): The path to the source file.
        destination_dir (str): The directory where the file should be copied.
        new_name (str): The new name for the file in the destination directory.

    Returns:
        str: The path to the renamed file in the destination directory.
    """
    # Copy the file to the destination directory
    shutil.copy(source_path, destination_dir)

    # Extract the file extension
    file_extension = os.path.splitext(source_path)[1]

    # Define the destination file path
    destination_file = os.path.join(destination_dir, f"{new_name}{file_extension}")

    # Check if the destination file already exists
    if os.path.exists(destination_file):
        # Ask the user if they want to delete the existing file
        user_input = input(f"File {destination_file} already exists. Do you want to delete it and proceed? (yes/no): ").strip().lower()
        if user_input == 'yes':
            os.remove(destination_file)
            print(f"Deleted existing file: {destination_file}")
        else:
            print("Operation cancelled by the user.")
            return None

    # Rename the copied file to the new name with the original extension
    os.rename(os.path.join(destination_dir, os.path.basename(source_path)), destination_file)
    print(f"File renamed to: {destination_file}")

    return destination_file

In [6]:

book_id = create_book_id(book_name, main_db)
working_folder = f"./data/srcs/{book_id}/"






In [ ]:
# Create the uploads directory if it doesn't exist
os.makedirs(working_folder + "uploads", exist_ok=True)

copy_and_rename_file(book_source, working_folder+"uploads", "source_doc")

In [32]:
Item = Query()

In [42]:
main_db.insert({"uid":book_id,"name": book_name, "folder": working_folder})

1

In [28]:
copy_and_rename_file(highlights_source, working_folder+"uploads", "highlights_doc")

File renamed to: ./data/srcs/the-invisible-empire/uploads\highlights_doc.html


'./data/srcs/the-invisible-empire/uploads\\highlights_doc.html'

In [43]:
highlights = KindleHTMLParser(working_folder + "uploads/highlights_doc.html").highlights
kindle_highlights_db.insert({"uid":book_id, 'kindle_highlights': highlights})

1

In [44]:
book_item_highlights = kindle_highlights_db.search(Item.uid == book_id)[0]
print(book_item_highlights["kindle_highlights"])

[{'title': '1 BOUNTY', 'highlights': ['A single gram of the stale-smelling yellow grimy film on our teeth, good old plaque, has approximately 1011 bacteria, which is about the same number as that of all the humans that have ever lived.', 'The effect of this interdependence is deeply significant as it sets the tone of relationships for all life on \xa0 Earth.', 'The problem with viruses is that they do not fit into any of the conventionally accepted domains of life— they are neither archaea, eukaryotes nor prokaryotes.']}, {'title': '4 THE VIRUS IS US', 'highlights': ['In the nearly two decades since the Human Genome Project, scientists have identified more than fifty distinct Human ERVs (or HERV) ‘families’ in human DNA. Of these, the HERV-L family is considered the oldest, and is estimated to have invaded the genome of an ancestor of all mammals some 150 million years ago. This was a momentous development for all modern mammals because it caused mammals to split into two distinct line

In [7]:
Doc = Query()
item = Doc.name == book_name
print(item)

QueryImpl('==', ('name',), 'The Invisible Empire')


In [11]:
from ebooklib import epub

book = epub.read_epub(book_source)

# Function to save content to a file
def save_file(file_path, content):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, 'wb') as f:
        f.write(content)

# Output directory
output_dir = working_folder + "unbundled_epub"
file_details = []
# Process each item in the EPUB
for item in book.get_items():
    file_path = os.path.join(output_dir, item.file_name)
    file_name = os.path.basename(item.file_name)
    file_ext = os.path.splitext(file_name)[1][1:]
    file_details.append({'file_name':file_name, 'file_ext':file_ext, 'file_path':item.file_name})
    save_file(file_path, item.content)

# Save the file details to the database
main_db.upsert({'children_file_details':file_details}, Doc.uid == book_id)
print(f"EPUB unbundled successfully into {output_dir}")

d:\Projects\Fun\summarizer\.venv\Lib\site-packages\ebooklib\epub.py:1395: UserWarning: In the future version we will turn default option ignore_ncx to True.
  warnings.warn('In the future version we will turn default option ignore_ncx to True.')
d:\Projects\Fun\summarizer\.venv\Lib\site-packages\ebooklib\epub.py:1423: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/xmlns:rootfile[@media-type]'
  for root_file in tree.findall('//xmlns:rootfile[@media-type]', namespaces={'xmlns': NAMESPACES['CONTAINERNS']}):


EPUB unbundled successfully into ./data/srcs/the-invisible-empire/unbundled_epub


In [12]:
#get the table of contents of the book
import ebooklib

def get_toc_details(book):
    toc = book.toc
    toc_details = []
    for item in toc:
        if isinstance(item, ebooklib.epub.Link):
            toc_item ={"type": "link", "href": item.href, "title": item.title, "uid": item.uid}

        elif isinstance(item, tuple) and isinstance(item[0], ebooklib.epub.Section):
            toc_item = {"type": "section", "title": item[0].title, "links": []}
            for link in item[1]:
                toc_item["links"].append({"type": "link", "href": link.href, "title": link.title, "uid": link.uid})
        toc_details.append(toc_item)
    return toc_details
toc = get_toc_details(book)
main_db.upsert({'toc':toc}, Doc.uid == book_id)
print(toc)


[{'type': 'link', 'href': 'xhtml/cover.xhtml', 'title': 'Cover', 'uid': 'cover'}, {'type': 'link', 'href': 'xhtml/toc.xhtml', 'title': 'Contents', 'uid': 'html-toc'}, {'type': 'link', 'href': 'xhtml/c001.xhtml', 'title': '1 BOUNTY', 'uid': 'c001'}, {'type': 'link', 'href': 'xhtml/c002.xhtml', 'title': '2 A WHOLE NEW WORLD', 'uid': 'c002'}, {'type': 'link', 'href': 'xhtml/c003.xhtml', 'title': '3 SUPERSIZE ME', 'uid': 'c003'}, {'type': 'link', 'href': 'xhtml/c004.xhtml', 'title': '4 THE VIRUS IS US', 'uid': 'c004'}, {'type': 'link', 'href': 'xhtml/c005.xhtml', 'title': '5 A DEEP CONTROL', 'uid': 'c005'}, {'type': 'link', 'href': 'xhtml/c006.xhtml', 'title': '6 INVADERS, HITCH-HIKERS, SENTINELS, KILLERS', 'uid': 'c006'}, {'type': 'link', 'href': 'xhtml/c007.xhtml', 'title': '7 A SPOTTY HISTORY OF THE SPECKLED MONSTER', 'uid': 'c007'}, {'type': 'link', 'href': 'xhtml/c008.xhtml', 'title': '8 GUT FEELING', 'uid': 'c008'}, {'type': 'link', 'href': 'xhtml/c009.xhtml', 'title': '9 A VIRUS VAN

In [14]:
from bs4 import BeautifulSoup
def extract_clean_content_from_html(html_loc):
    with open(html_loc, 'rb') as f:
        content = f.read()
        if content:
            soup = BeautifulSoup(content.decode('utf-8'), 'html.parser')
            return soup.get_text()  
        return None

for item in toc:
    html_loc = working_folder + "unbundled_epub/" + item["href"]
    content = extract_clean_content_from_html(html_loc)



In [174]:
import os.path
from llama_index.core import (
    VectorStoreIndex,
    StorageContext,
    Document,
    load_index_from_storage,
    Settings
)
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-large"
)

Settings.llm = OpenAI(model="gpt-4o")



In [179]:
from llama_index.core import VectorStoreIndex, StorageContext, Document
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

# Initialize Chroma client and collection
db = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = db.get_or_create_collection("books")

# Create vector store and storage context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Initialize index
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    storage_context=storage_context,
    embed_model=Settings.embed_model,
)

# Insert document and get metadata
def process_and_insert(item):
    html_loc = f"{working_folder}unbundled_epub/{item['href']}"
    content = extract_clean_content_from_html(html_loc)
    
    doc = Document(
        text=content,
        id_=item["href"],  # Explicit document ID
        metadata={
            "href": item["href"],
            "chapter_title": item["title"],
            "book_title": book_name
        }
    )
    
    # Insert and get node with metadata
    index.insert(doc)

    
    # Return document metadata and Chroma ID
    return {
        "doc_id": doc.doc_id,
        "metadata": doc.metadata,
    }




In [ ]:
# Process documents and collect metadata
document_metadata = []
for item in toc[:3]:
    meta = process_and_insert(item)
    document_metadata.append(meta)
    print(f"Inserted document {meta['doc_id']} with metadata: {meta['metadata']}")

In [178]:
import chromadb
client = chromadb.PersistentClient(path="./chroma_db")  # Or HttpClient()
# collection = client.delete_collection(name="books")
collection = client.get_or_create_collection("books")
collection.peek()

{'ids': ['39b2c9ce-6c0f-4521-ae47-ffcc0e8dfa0a',
  'e41116db-5076-4949-9eff-e5d75fc12c9f',
  'f25e4ad9-d451-4c01-a00b-a485d26e9a5b',
  '6a87da6a-8b71-47f8-aa7b-ff5c6e68a480',
  'ae9b188d-100f-4cee-8ac6-f3adea864820',
  '3721c289-a476-4f20-9373-505ac33d9797',
  '25783b33-4b57-4b9f-98cc-4464fa732f63',
  'b3ca8284-9dd5-4276-b7ec-dccd0cbefcdf'],
 'embeddings': array([[-0.00201614,  0.00476012, -0.00131555, ...,  0.00254743,
         -0.0137705 , -0.01122502],
        [-0.00972714, -0.01416044, -0.012885  , ..., -0.00082308,
         -0.01578163, -0.01804053],
        [ 0.01503754, -0.01582511, -0.0088765 , ...,  0.00446286,
         -0.01535749,  0.00192789],
        ...,
        [ 0.00437074, -0.01110473, -0.00946147, ...,  0.00052331,
         -0.00305571, -0.01433196],
        [-0.00399988, -0.02268805, -0.01089129, ...,  0.00216224,
         -0.01537594, -0.02174841],
        [-0.00807218, -0.00211339, -0.0064963 , ...,  0.00482403,
         -0.02176052, -0.00708307]]),
 'documents': [

In [180]:
from typing import List
from llama_index.core.vector_stores import MetadataFilters
from llama_index.core.schema import Document
from llama_index.core.retrievers import (
    VectorIndexRetriever,
)
from llama_index.core import QueryBundle

def vector_retriever_with_metadata_filters(index: any, query: str, metadata_filters: MetadataFilters = None) -> List[Document]:
    """
    Retrieve nodes from the index using specified metadata filters and a query.
    
    Args:
        index (any): The index object that supports retrieval operations.
        query (str): The search query string used to retrieve relevant documents.
        filters (MetadataFilters, optional): A collection of metadata filters to apply during retrieval.

    Returns:
        List[Document]: A list of documents matching the query and filters.
    """

    vector_retriever = VectorIndexRetriever(index=index, similarity_top_k=2, filters=metadata_filters)

    # vector query engine
    retriever = vector_retriever.retrieve(QueryBundle(query))


    return retriever



In [181]:
from llama_index.core.vector_stores.types import MetadataFilters, ExactMatchFilter, FilterCondition
query = "What summary?"

# Define multiple href values
href_values = ["xhtml/c001.xhtml", "xhtml/c002.xhtml", "xhtml/c003.xhtml"]

# Create a list of ExactMatchFilter for each href value
filters_list = [ExactMatchFilter(key="href", value=href) for href in href_values]

# Combine the filters using an OR condition
filters = MetadataFilters(filters=filters_list, condition=FilterCondition.OR)

# Use the filters in your retriever
response = vector_retriever_with_metadata_filters(index, query, filters)

print(response)


[NodeWithScore(node=TextNode(id_='ae9b188d-100f-4cee-8ac6-f3adea864820', embedding=None, metadata={'href': 'xhtml/c001.xhtml', 'chapter_title': '1 BOUNTY', 'book_title': 'The Invisible Empire'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='xhtml/c001.xhtml', node_type='4', metadata={'href': 'xhtml/c001.xhtml', 'chapter_title': '1 BOUNTY', 'book_title': 'The Invisible Empire'}, hash='0fa5a026d9a8b09da19260916cc9e85b4bed7e9d05b1bf573d336eee468ac754'), <NodeRelationship.PREVIOUS: '2'>: RelatedNodeInfo(node_id='6a87da6a-8b71-47f8-aa7b-ff5c6e68a480', node_type='1', metadata={'href': 'xhtml/c001.xhtml', 'chapter_title': '1 BOUNTY', 'book_title': 'The Invisible Empire'}, hash='a3eb8a421019f4386265926f7252d2feafd85d91c9d806bbf1fc472ee013f073'), <NodeRelationship.NEXT: '3'>: RelatedNodeInfo(node_id='3721c289-a476-4f20-9373-505ac33d9797', node_type='1', metadata={}, hash='b3ccedb0eca4de0c34eacc28ee1c975463

In [47]:
#custom query engine

from llama_index.core import get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

def simple_answer_query(index: any, query: str, metadata_filters: MetadataFilters = None, output_format: classmethod = None, response_mode: str = "compact") -> List[Document]:        
    # configure retriever
    retriever = VectorIndexRetriever(
        index=index,
        filters=metadata_filters
    )

    # configure response synthesizer
    response_synthesizer = get_response_synthesizer(output_cls=output_format, response_mode=response_mode)

    # assemble query engine
    query_engine = RetrieverQueryEngine(
        retriever=retriever,
        response_synthesizer=response_synthesizer,
    )

    # query
    response = query_engine.query(query)
    return response

In [183]:
#custom query engine

from llama_index.core import get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core import PromptTemplate
from IPython.display import Markdown, display

# define prompt viewing function
def display_prompt_dict(prompts_dict):
    for k, p in prompts_dict.items():
        text_md = f"**Prompt Key**: {k}" f"**Text:** "
        display(Markdown(text_md))
        print(p.get_template())
        display(Markdown(""))

def summarizer_with_quotations(index: any, query: str, quotes: List[str] = None, metadata_filters: MetadataFilters = None, output_format: classmethod = None, response_mode: str = "tree_summarize") -> List[Document]:        
    # configure retriever
    retriever = VectorIndexRetriever(
        index=index,
        filters=metadata_filters,
        similarity_top_k=10
    )

    summary_prompt_template = """
    Below is a section of a book along with important quotes from throughout the book.
    
    Book Content:
    {context_str}
    
    Important Quotes:
    {quotes}
    
    Task: Create a coherent summary of this section that:
    1. Synthesizes the main ideas and themes
    2. Naturally incorporates relevant quotes where they support key points
    3. Maintains a flowing narrative structure
    4. Connects ideas across different parts of the book
    
    Summary:
    """

    prompt_tmpl = PromptTemplate(
        summary_prompt_template,
    )
    formatted_quotes = "\n".join([f"• {quote}" for quote in quotes])
    template_with_variables = prompt_tmpl.partial_format(quotes=formatted_quotes)


    # configure response synthesizer
    response_synthesizer = get_response_synthesizer(output_cls=output_format, response_mode=response_mode, text_qa_template=template_with_variables, streaming=True, verbose=True)

    # assemble query engine
    query_engine = RetrieverQueryEngine(
        retriever=retriever,
        response_synthesizer=response_synthesizer,
        node_postprocessors=[],
    )

    response = query_engine.query(query)
    
    return response

In [187]:

Item = Query()

# use tinydb to get the highlights for a chapter
def get_highlights_for_chapter(chapter_title, book_id):
    book_item_highlights = kindle_highlights_db.search(Item.uid == book_id)[0]
    highlights = book_item_highlights["kindle_highlights"]
    chapter_highlights = find_highlights_for_chapter(chapter_title, highlights)
    return chapter_highlights

# get the highlights for the first chapter
book_id = "the-invisible-empire"
chapter_title = "1 BOUNTY"
chapter_highlights = get_highlights_for_chapter(chapter_title, book_id)
print(chapter_highlights)

['A single gram of the stale-smelling yellow grimy film on our teeth, good old plaque, has approximately 1011 bacteria, which is about the same number as that of all the humans that have ever lived.', 'The effect of this interdependence is deeply significant as it sets the tone of relationships for all life on \xa0 Earth.', 'The problem with viruses is that they do not fit into any of the conventionally accepted domains of life— they are neither archaea, eukaryotes nor prokaryotes.']


In [190]:
from pydantic import BaseModel
from llama_index.core.vector_stores.types import MetadataFilters, ExactMatchFilter, FilterCondition, MetadataFilter
import json

class InfoPointer(BaseModel):
    pointer: str
    related_highlight: str = None

class InfoPoints(BaseModel):
    subtitle: str
    pointers: List[InfoPointer]

class Information(BaseModel):
    title: str
    info_points: List[InfoPoints]

query = "Create a comprehensive summary of the chapter that incorporates the provided quotes naturally."
additional_info = chapter_highlights
# Define multiple href values
href_values = ["xhtml/c001.xhtml"]

# Create a list of ExactMatchFilter for each href value
filters_list = [ExactMatchFilter(key="href", value=href) for href in href_values]

# Combine the filters using an OR condition
filters = MetadataFilters(filters=filters_list, condition=FilterCondition.OR)

llama_response = summarizer_with_quotations(index=index, query=query, quotes=additional_info, metadata_filters=filters, output_format=Information)

print(llama_response)
print(dir(llama_response))
json_response = json.loads(llama_response.response)
# print json in a pretty format
print(json.dumps(json_response, indent=2))



{"title":"The Invisible Empire - Chapter 1: BOUNTY","info_points":[{"subtitle":"Microbial Abundance and Diversity","pointers":[{"pointer":"There are 100 million times as many bacteria in the oceans as there are stars in the known universe.","related_highlight":null},{"pointer":"A single teaspoon of soil contains as many microbes as the human population of Africa.","related_highlight":null},{"pointer":"A single gram of plaque on teeth has approximately 10^11 bacteria, about the same number as all humans that have ever lived.","related_highlight":"A single gram of the stale-smelling yellow grimy film on our teeth, good old plaque, has approximately 1011 bacteria, which is about the same number as that of all the humans that have ever lived."}]},{"subtitle":"Microbial Ecosystems and Interdependence","pointers":[{"pointer":"Microbes form complex ecosystems in soil and water, with producers, predators, prey, and parasites.","related_highlight":null},{"pointer":"Communities of microbes are d

In [48]:


from pydantic import BaseModel

class InfoPoints(BaseModel):
    subtitle: str
    pointers: List[str]

class Information(BaseModel):
    title: str
    info_points: List[InfoPoints]

query = "How did microscope play a role in learning about viruses?"
# Define multiple href values
href_values = ["xhtml/c001.xhtml", "xhtml/c002.xhtml", "xhtml/c003.xhtml"]

# Create a list of ExactMatchFilter for each href value
filters_list = [ExactMatchFilter(key="href", value=href) for href in href_values]

# Combine the filters using an OR condition
filters = MetadataFilters(filters=filters_list, condition=FilterCondition.OR)
response = custom_query_engine(index, query, filters, output_format=Information)
print(response)

{"title":"Role of Microscope in Learning About Viruses","info_points":[{"subtitle":"Microscope Invention","pointers":["The invention of the microscope allowed scientists to observe minutiae and discuss them, although initially, it was challenging to differentiate between various microorganisms."]},{"subtitle":"Limitations with Viruses","pointers":["Viruses are much smaller than the wavelength of light, making them invisible under traditional light microscopes.","Scientists use electrons instead of light to visualize viruses, as electrons are smaller than the wavelength of light."]},{"subtitle":"Advanced Techniques","pointers":["Electron microscopy and x-ray crystallography are used to determine the shapes, sizes, and structures of viruses.","These techniques provide a 3D rendering of viruses without involving light, hence no color is observed."]}]}


In [108]:
from pydantic import BaseModel
from llama_index.core.query_engine import RetrieverQueryEngine

class SummaryPoint(BaseModel):
    subtitle: str
    pointers: List[str]

class Summary(BaseModel):
    summary_content: str
    points: List[SummaryPoint]

from llama_index.core import get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine


# configure retriever
retriever = VectorIndexRetriever(
    index=index
)

# configure response synthesizer
response_synthesizer = get_response_synthesizer(
    response_mode="tree_summarize",
    output_cls=Summary
)

# assemble query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)

# query
response = query_engine.query("Give the summary of the book")
print(response)


{"summary_content":"\"Invisible Empire: The Natural History Of Viruses\" explores the complex and often misunderstood role of viruses in the natural world. The book aims to challenge the negative perception of viruses by highlighting their essential contributions to ecosystems and human life. Through eleven selected stories, the author delves into the ecological and beneficial aspects of viruses, offering a narrative that contrasts with the typical portrayal of viruses as mere pathogens. The book is a result of the author's extensive research and discussions with experts, aiming to provide a nuanced perspective on the virus-human relationship.","points":[{"subtitle":"Introduction to Viruses","pointers":["Viruses are often viewed negatively due to their association with diseases.","The book seeks to present a balanced view of viruses as integral parts of ecosystems."]},{"subtitle":"The Role of Viruses","pointers":["Viruses contribute to ecological balance and biodiversity.","They play a

In [78]:
from typing import List, Dict, Any
from llama_index.core.indices.base import BaseIndex

def get_nodes_with_ref_doc_id(index: BaseIndex, target_ref_doc_id: str) -> List[Dict[str, Any]]:
    """
    Retrieve nodes associated with a specific reference document ID from the index.

    This function fetches all reference document information from the index's docstore,
    checks if the specified reference document ID exists, and retrieves the nodes
    associated with it. Each node is converted to a dictionary before being returned.

    Args:
        index (BaseIndex): The index object containing the docstore and retrieval methods.
        target_ref_doc_id (str): The reference document ID for which nodes need to be retrieved.

    Returns:
        List[Dict[str, Any]]: A list of dictionaries representing the nodes associated
        with the given reference document ID. If no nodes are found, an empty list is returned.
    """
    ref_doc_info = index.docstore.get_all_ref_doc_info()

    nodes = []
    if target_ref_doc_id in ref_doc_info:
        node_ids = ref_doc_info[target_ref_doc_id].node_ids  # List of node IDs
        print(f"Nodes for {target_ref_doc_id}: {node_ids}")
        for node_id in node_ids:
            node = index.docstore.get_node(node_id)
            nodes.append(node.to_dict())
    else:
        print(f"No nodes found for ref_doc_id: {target_ref_doc_id}")

    return nodes

Nodes for id_cover: ['9b5c8709-2d0f-4576-bdb5-d8e975693164']
[{'id_': '9b5c8709-2d0f-4576-bdb5-d8e975693164', 'embedding': None, 'metadata': {'href': 'xhtml/cover.xhtml', 'title': 'Cover'}, 'excluded_embed_metadata_keys': [], 'excluded_llm_metadata_keys': [], 'relationships': {'1': {'node_id': 'id_cover', 'node_type': '4', 'metadata': {'href': 'xhtml/cover.xhtml', 'title': 'Cover'}, 'hash': '80f344cfe9ed9daaee4fd89cd66fc8d916e2cd1c9459da063e41eef84367f97a', 'class_name': 'RelatedNodeInfo'}}, 'metadata_template': '{key}: {value}', 'metadata_separator': '\n', 'text': 'Invisible Empire: The Natural History Of Viruses', 'mimetype': 'text/plain', 'start_char_idx': 0, 'end_char_idx': 48, 'metadata_seperator': '\n', 'text_template': '{metadata_str}\n\n{content}', 'class_name': 'TextNode'}]


In [ ]:
nodes = get_nodes_with_ref_doc_id(index, "id_cover")
print(nodes)